# H13 · Size and sector: who carries the country effect?

> **Hypothesis H13:** The (negative) link between development and ROA is weaker for the largest companies in each country: big firms are less tied to their home country's conditions.

| | |
|---|---|
| **Why we ask** | The country effect on ROA is small on average (H0). An average can hide a strong effect in some firms and none in others. Large listed companies sell abroad, borrow internationally and can lobby; small ones depend on the domestic economy. If so, the country effect should shrink with size. |
| **Prediction** | **Positive** interaction between the development index and relative size (the development effect is negative for small firms and closer to zero for large ones). |
| **Data** | Yahoo company-years. Size = percentile of total assets within country and fiscal year (0 = smallest, 1 = largest); this avoids exchange-rate problems. |
| **Primary test (fixed before looking at the result)** | `ROA (pp) ~ development index × size rank + sector + fiscal year`; the test is the coefficient on the product (development index × size rank). Errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra="", controls=CONTROLS, data=None):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy if data is None else data), outcome=outcome, extra=extra, B=B, controls=controls)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · ROA by country type and size
Median ROA in a grid of development terciles (of the 34 countries) and size terciles (within country).

In [2]:
fy["dev_x_size"] = fy.dev_z * fy.size_rank
fy["size_t"] = pd.cut(fy.size_rank, [0, 1/3, 2/3, 1.0001], labels=["small", "medium", "large"], include_lowest=True)
fy["dev_t"] = pd.qcut(fy.iso3.map(country["dev_idx"]), 3, labels=["least developed", "middle", "most developed"])
grid = fy.pivot_table(index="size_t", columns="dev_t", values="roa_pp", aggfunc="median", observed=True)
cnt = fy.pivot_table(index="size_t", columns="dev_t", values="roa_pp", aggfunc="size", observed=True)
fig = px.imshow(grid, text_auto=".1f", color_continuous_scale="RdBu", color_continuous_midpoint=grid.values.mean(), title="Median ROA (pp) by size tercile and country-development tercile", aspect="auto")
fig.update_layout(height=360); fig.show()
print("Number of firm-years per cell:"); print(cnt.to_string())

Number of firm-years per cell:
dev_t   least developed  middle  most developed
size_t                                         
small              1249    1019            1105
medium             1259    1027            1116
large              1273    1042            1130


## 2 · Primary test

In [3]:
primary = wild_cluster("dev_x_size", fy, outcome="roa_pp", extra="dev_z", B=1999)
main = wild_cluster("dev_z", fy, outcome="roa_pp", B=999)
print(f"Development effect at the average size (H0): {main['coef']:+.2f} pp per +1 SD.")
print(f"Interaction: the effect changes by {primary['coef']:+.2f} pp per +1 SD for the largest vs the smallest firm in the country (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}).")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f} | {primary['clusters']} countries, {primary['n']:,} company-years")

Development effect at the average size (H0): -1.12 pp per +1 SD.
Interaction: the effect changes by +3.10 pp per +1 SD for the largest vs the smallest firm in the country (95% CI +1.31 to +4.89).
p (cluster-robust) = 0.0018 | p (wild bootstrap) = 0.0075 | 34 countries, 10,220 company-years


### The development effect within each size group
The H0 model estimated separately for small, medium and large companies (wild bootstrap, 999 draws).

In [4]:
rows = []
for lab, g in fy.groupby("size_t", observed=True):
    r = wild_cluster("dev_z", g, outcome="roa_pp", B=999)
    rows.append({"size": lab, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "firm-years": r["n"], "median ROA": g.roa_pp.median()})
by_size = pd.DataFrame(rows).set_index("size")
fig = px.scatter(by_size.reset_index(), x="coef", y="size", error_x=by_size.ci_high - by_size.coef, error_x_minus=by_size.coef - by_size.ci_low, title="Development → ROA (pp per +1 SD) by company size")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=320, yaxis_autorange="reversed"); fig.show()
by_size

,coef,ci_low,ci_high,p (wild),firm-years,median ROA
size,,,,,,
small,-2.007,-3.178,-0.835,0.007,3373,4.354
medium,-1.234,-1.768,-0.701,0.001,3402,4.563
large,-0.193,-0.407,0.021,0.103,3445,2.758


## 3 · By sector
The development effect estimated within each sector (size and year controls, wild bootstrap 499 draws), corrected for multiple testing across sectors. Do the sectors with the highest margins show the strongest country effect?

In [5]:
rows = []
for sct, g in fy.groupby("sector"):
    if g.iso3.nunique() < 15 or len(g) < 200:
        continue
    r = wild_cluster("dev_z", g, outcome="roa_pp", controls="size_rank + C(fiscal_year)", B=499)
    rows.append({"sector": sct, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "countries": r["clusters"], "firm-years": r["n"], "median ROA": g.roa_pp.median()})
by_sec = pd.DataFrame(rows).set_index("sector").sort_values("coef"); by_sec["q (BH)"] = bh(by_sec["p (wild)"])
fig = px.scatter(by_sec.reset_index(), x="coef", y="sector", error_x=by_sec.ci_high - by_sec.coef, error_x_minus=by_sec.coef - by_sec.ci_low, title="Development → ROA (pp per +1 SD) within each sector")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=470, yaxis_autorange="reversed"); fig.show()
rs, ps, ns = spearman(by_sec["median ROA"], by_sec.coef)
print(f"{len(by_sec)} sectors; negative in {int((by_sec.coef < 0).sum())}; q < 0.10 in {int((by_sec['q (BH)'] < 0.10).sum())}. Sector's typical ROA vs its country effect: ρ = {rs:+.2f} (n = {ns}).")
by_sec.round(3)

11 sectors; negative in 10; q < 0.10 in 5. Sector's typical ROA vs its country effect: ρ = -0.02 (n = 11).


,coef,ci_low,ci_high,p (wild),countries,firm-years,median ROA,q (BH)
sector,,,,,,,,
Healthcare,-2.365,-4.143,-0.588,0.012,30,850,4.448,0.026
Utilities,-2.360,-3.162,-1.559,0.002,30,750,2.582,0.011
Consumer Defensive,-2.209,-3.555,-0.862,0.010,33,1023,5.207,0.026
Technology,-1.647,-3.917,0.624,0.366,30,841,5.928,0.447
Real Estate,-1.470,-1.972,-0.967,0.002,31,988,2.776,0.011
Basic Materials,-1.391,-2.402,-0.380,0.010,30,948,4.509,0.026
Energy,-1.317,-3.152,0.517,0.272,30,577,4.972,0.374
Communication Services,-0.658,-1.717,0.401,0.272,30,799,3.619,0.374
Industrials,-0.389,-0.965,0.188,0.250,33,1240,4.604,0.374


### Institutions and size
The same interaction for the institutions index (secondary, not part of the verdict).

In [6]:
fy["inst_x_size"] = fy.inst_z * fy.size_rank
inter_inst = wild_cluster("inst_x_size", fy, outcome="roa_pp", extra="inst_z", B=999)
print(f"Institutions × size: {inter_inst['coef']:+.2f} pp (95% CI {inter_inst['ci_low']:+.2f} to {inter_inst['ci_high']:+.2f}), wild p = {inter_inst['p_wild']:.3f}")

Institutions × size: +2.98 pp (95% CI +1.11 to +4.86), wild p = 0.008


## 4 · Robustness
The interaction in seven samples (wild bootstrap, 499 draws each).

In [7]:
rob, rob_share = robustness("dev_x_size", "roa_pp", 1, extra="dev_z")
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="Development × size interaction (pp) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Predicted sign (positive) in {rob_share:.0%} of samples.")
rob

Predicted sign (positive) in 100% of samples.


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,3.098,1.308,4.889,0.004,34,10220,True
excluding financials & real estate,3.523,1.281,5.765,0.008,34,8063,True
excluding small-sample countries,2.914,1.080,4.748,0.014,29,9787,True
"excluding USA, China, Japan",3.386,1.662,5.109,0.012,31,9023,True
fiscal years 2023 and later,3.360,1.658,5.063,0.002,34,7603,True
EU countries only,5.057,0.504,9.609,0.102,21,5121,True
non-EU countries only,1.090,-1.080,3.259,0.456,13,5099,True


## 5 · Replication on ESEF

In [8]:
fe = prep.load_firms(con, country, source="ESEF"); fe["dev_x_size"] = fe.dev_z * fe.size_rank
rep = wild_cluster("dev_x_size", fe, outcome="roa_pp", extra="dev_z", controls="size_rank + C(fiscal_year)", B=1999)
rep_same = bool(np.sign(rep["coef"]) == 1)
print(f"ESEF ({rep['n']:,} company-years, {rep['clusters']} countries): interaction {rep['coef']:+.2f} pp (95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")

ESEF (5,312 company-years, 19 countries): interaction +6.06 pp (95% CI +2.43 to +9.69), wild p = 0.004


## 6 · Verdict

In [9]:
v = utils.verdict(primary["coef"], primary["p_wild"], 1, rob_share, primary["clusters"])
lines = [
 f"Primary: development × size interaction = {primary['coef']:+.2f} pp (effect at average size {main['coef']:+.2f} pp per SD), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} company-years.",
 "By size: " + "; ".join(f"{i} {r.coef:+.2f}" for i, r in by_size.iterrows()) + " pp per SD.",
 f"Robustness: predicted sign in {rob_share:.0%} of seven samples. ESEF replication: {rep['coef']:+.2f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"By sector: development effect negative in {int((by_sec.coef < 0).sum())} of {len(by_sec)} sectors; q < 0.10 in {int((by_sec['q (BH)'] < 0.10).sum())}. Institutions × size: {inter_inst['coef']:+.2f} (p = {inter_inst['p_wild']:.3f}).",
 f"VERDICT H13: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H13", id="H13", title="Size and sector: who carries the country effect", statement="The development–ROA link is weaker for large companies.",
                  expected_sign=1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp ROA, development × size rank interaction",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.13_size_and_sector_splits.ipynb", notes=lines)

Primary: development × size interaction = +3.10 pp (effect at average size -1.12 pp per SD), wild p = 0.007, 34 countries, 10,220 company-years.
By size: small -2.01; medium -1.23; large -0.19 pp per SD.
Robustness: predicted sign in 100% of seven samples. ESEF replication: +6.06 pp (wild p = 0.004), same direction.
By sector: development effect negative in 10 of 11 sectors; q < 0.10 in 5. Institutions × size: +2.98 (p = 0.008).
VERDICT H13: SUPPORTED


### Limits
- Size is relative *within* each country, so a "large" firm in a small market may be small globally; size and listing venue are not the same.
- Sector splits use small samples in some sectors and countries, and cannot say anything about sectors absent from a country.
- ESEF has no sector information and a different size distribution.
- Association, not causation.

In [10]:
con.close()